# Sanskrit → English NMT (NLU Assignment)

Fine-tune **`facebook/nllb-200-distilled-600M`** with LoRA on the provided corpus.

- Source language: `san_Deva` (Sanskrit, Devanagari) — officially supported by NLLB-200
- Target language: `eng_Latn` (English)
- Hardware: Apple MPS · LoRA only (no QLoRA on macOS)

Re-run top-to-bottom. To predict a new test set, replace `dl_dataset/test_sa_1000.csv` and re-run from Section 11.

## 1. Install dependencies

In [ ]:
!pip install -q torch transformers datasets peft accelerate sentencepiece nltk bert-score pandas matplotlib tqdm

## 2. Imports, seeds, and configuration

In [ ]:
import random
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import torch
from bert_score import score as bert_score_fn
from datasets import Dataset
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from peft import LoraConfig, TaskType, get_peft_model
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed as hf_set_seed,
)

warnings.filterwarnings("ignore")
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); hf_set_seed(SEED)

PROJECT_DIR = Path(".").resolve()
DATA_DIR = PROJECT_DIR / "dl_dataset"
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
METRICS_DIR = OUTPUT_DIR / "metrics"
for d in [OUTPUT_DIR, CHECKPOINT_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FILE_MAP = {
    "train": ("train_sa_10000.csv", "train_en_10000.csv"),
    "dev":   ("dev_sa_1000.csv",   "dev_en_1000.csv"),
    "test":  ("test_sa_1000.csv",  "test_en_1000.csv"),
}
ID_COL, SRC_COL, TGT_COL = "Source_id", "Sentence_sa", "Sentence_en"

MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "san_Deva"
TGT_LANG = "eng_Latn"

NUM_EPOCHS = 6              # 4 prior + 2 extra; early stopping may stop sooner
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 8
NUM_BEAMS = 5
LENGTH_PENALTY = 0.8
NO_REPEAT_NGRAM_SIZE = 3
EARLY_STOPPING_PATIENCE = 2
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "out_proj"]  # wider adapter (was q,v only)

DEVICE = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}  |  Model: {MODEL_NAME}")

## 3. Load and validate CSV files

In [ ]:
import unicodedata

def normalize_text(text: str) -> str:
    return unicodedata.normalize("NFC", " ".join(str(text).strip().split()))


def load_parallel_split(split: str, require_targets: bool = True) -> pd.DataFrame:
    sa_file, en_file = FILE_MAP[split]
    sa_path, en_path = DATA_DIR / sa_file, DATA_DIR / en_file
    if not sa_path.exists():
        raise FileNotFoundError(f"Missing: {sa_path}")

    sa_df = pd.read_csv(sa_path, encoding="utf-8-sig")
    en_df = pd.read_csv(en_path, encoding="utf-8-sig") if en_path.exists() else None

    for name, df, cols in [(sa_file, sa_df, [ID_COL, SRC_COL])]:
        if missing := [c for c in cols if c not in df.columns]:
            raise ValueError(f"{name} missing columns: {missing}")
    if en_df is not None and (missing := [c for c in [ID_COL, TGT_COL] if c not in en_df.columns]):
        raise ValueError(f"{en_file} missing columns: {missing}")

    sa_df = sa_df.drop_duplicates(subset=[ID_COL], keep="first")
    sa_df = sa_df.rename(columns={SRC_COL: "source"})
    sa_df["source"] = sa_df["source"].astype(str).map(normalize_text)

    if en_df is not None:
        en_df = en_df.drop_duplicates(subset=[ID_COL], keep="first")
        en_df = en_df.rename(columns={TGT_COL: "target"})
        en_df["target"] = en_df["target"].astype(str).map(normalize_text)
        merged = sa_df.merge(en_df[[ID_COL, "target"]], on=ID_COL, how="inner")
    else:
        merged = sa_df.assign(target="")

    merged = merged[(merged["source"] != "")]
    if require_targets:
        merged = merged[(merged["target"] != "")]
    return merged.sort_values(ID_COL).reset_index(drop=True)


train_df = load_parallel_split("train")
dev_df = load_parallel_split("dev")
test_df = load_parallel_split("test")

print(f"train={len(train_df)}  dev={len(dev_df)}  test={len(test_df)}")
display(train_df.head(3))

## 4. Load NLLB tokenizer & model (once)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.src_lang = SRC_LANG
TGT_TOKEN_ID = tokenizer.convert_tokens_to_ids(TGT_LANG)

# Sequence lengths from 95th percentile (NLLB tokenizer)
def p95_token_len(texts, sample=2000):
    lens = [len(tokenizer.encode(t, add_special_tokens=True)) for t in texts[:sample]]
    return int(np.percentile(lens, 95))

MAX_SOURCE_LENGTH = min(256, max(128, p95_token_len(train_df["source"].tolist() + dev_df["source"].tolist()) + 8))
MAX_TARGET_LENGTH = min(256, max(128, p95_token_len(train_df["target"].tolist() + dev_df["target"].tolist()) + 8))
print(f"max_source_length={MAX_SOURCE_LENGTH}, max_target_length={MAX_TARGET_LENGTH}")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.generation_config.forced_bos_token_id = TGT_TOKEN_ID  # use generation_config, not model.config
model.to(DEVICE)
print(f"NLLB: {SRC_LANG} → {TGT_LANG}  (forced_bos_id={TGT_TOKEN_ID})")

## 5. Tokenize datasets

In [ ]:
def preprocess(examples):
    tokenizer.src_lang = SRC_LANG
    model_inputs = tokenizer(examples["source"], max_length=MAX_SOURCE_LENGTH, truncation=True)
    labels = tokenizer(text_target=examples["target"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = Dataset.from_pandas(train_df[[ID_COL, "source", "target"]])
dev_ds   = Dataset.from_pandas(dev_df[[ID_COL, "source", "target"]])

tokenized_train = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
tokenized_dev   = dev_ds.map(preprocess, batched=True, remove_columns=dev_ds.column_names)
print(tokenized_train)

## 6. Apply LoRA

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS, bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

## 7. Train (Seq2SeqTrainer + early stopping on dev BLEU)

In [ ]:
def compute_corpus_bleu(references, hypotheses):
    """NLTK tokenization + method1 (standard for assignment reporting)."""
    refs = [[nltk.word_tokenize(r.lower())] for r in references]
    hyps = [nltk.word_tokenize(h.lower()) for h in hypotheses]
    return float(corpus_bleu(refs, hyps, smoothing_function=SmoothingFunction().method1))


data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR / "runs_v2"),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    num_train_epochs=NUM_EPOCHS,
    predict_with_generate=False,       # faster epochs; BLEU measured at inference
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=25,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
    fp16=False,
    dataloader_num_workers=0,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

train_result = trainer.train()
trainer.save_model(str(CHECKPOINT_DIR / "best"))
tokenizer.save_pretrained(str(CHECKPOINT_DIR / "best"))
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Final training loss: {train_result.training_loss:.4f}")

## 8. Training curves

In [ ]:
log_history = trainer.state.log_history
train_losses = [(e["step"], e["loss"]) for e in log_history if "loss" in e]
eval_entries = [e for e in log_history if "eval_loss" in e]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if train_losses:
    steps, losses = zip(*train_losses)
    axes[0].plot(steps, losses); axes[0].set(xlabel="Step", ylabel="Loss", title="Training Loss")
if eval_entries:
    epochs = range(1, len(eval_entries) + 1)
    axes[1].plot(epochs, [e["eval_loss"] for e in eval_entries], marker="o", label="eval loss")
    if any("eval_bleu" in e for e in eval_entries):
        ax2 = axes[1].twinx()
        ax2.plot(epochs, [e.get("eval_bleu", 0) for e in eval_entries], marker="s", color="green", label="eval BLEU")
        ax2.set_ylabel("BLEU"); ax2.legend(loc="upper right")
    axes[1].set(xlabel="Epoch", ylabel="Eval Loss", title="Validation")
    axes[1].legend(loc="upper left")
plt.tight_layout()
plt.savefig(METRICS_DIR / "training_curves.png", dpi=150); plt.show()

## 9. Dev evaluation (BLEU + BERTScore F1)

In [ ]:
def generate_translations(sources):
    model.eval()
    preds = []
    gen_cfg = dict(
        max_length=MAX_TARGET_LENGTH, num_beams=NUM_BEAMS,
        forced_bos_token_id=TGT_TOKEN_ID, early_stopping=True,
        length_penalty=LENGTH_PENALTY, no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
    )
    with torch.no_grad():
        for i in tqdm(range(0, len(sources), EVAL_BATCH_SIZE), desc="Generating"):
            batch = sources[i : i + EVAL_BATCH_SIZE]
            tokenizer.src_lang = SRC_LANG
            enc = tokenizer(batch, return_tensors="pt", padding=True,
                            truncation=True, max_length=MAX_SOURCE_LENGTH)
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            out = model.generate(**enc, **gen_cfg)
            preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return [p.strip() for p in preds]


def compute_bertscore_f1(references, hypotheses):
    _, _, f1 = bert_score_fn(hypotheses, references, lang="en",
                             rescale_with_baseline=True, verbose=False, device="cpu")
    return float(f1.mean().item())

t0 = time.time()
dev_preds = generate_translations(dev_df["source"].tolist())
dev_time = time.time() - t0
dev_refs = dev_df["target"].tolist()
dev_bleu = compute_corpus_bleu(dev_refs, dev_preds)
dev_bert = compute_bertscore_f1(dev_refs, dev_preds)

print(f"Dev BLEU:         {dev_bleu:.4f}")
print(f"Dev BERTScore F1: {dev_bert:.4f}")
print(f"Dev infer time:   {dev_time:.1f}s  ({dev_time/len(dev_preds)*1000:.0f} ms/sample)")
print(f"Total params:     {total_params:,}  |  Trainable: {trainable_params:,}")

## 10. Test predictions → submission.csv

In [ ]:
test_sa = load_parallel_split("test", require_targets=False).sort_values(ID_COL)

t0 = time.time()
test_preds = generate_translations(test_sa["source"].tolist())
test_time = time.time() - t0

submission = pd.DataFrame({"Source id": test_sa[ID_COL].astype(int), "Sentence en": test_preds})
submission.to_csv(PROJECT_DIR / "submission.csv", index=False, encoding="utf-8")
print(f"Saved submission.csv ({len(submission)} rows)")
display(submission.head())

test_refs = test_df.set_index(ID_COL)["target"]
aligned_refs = [test_refs.get(sid, "") for sid in test_sa[ID_COL]]
test_bleu = test_bert = None
if any(r.strip() for r in aligned_refs):
    test_bleu = compute_corpus_bleu(aligned_refs, test_preds)
    test_bert = compute_bertscore_f1(aligned_refs, test_preds)
    print(f"Local test BLEU: {test_bleu:.4f}")
    print(f"Local test BERTScore F1: {test_bert:.4f}")
print(f"Test infer time: {test_time:.1f}s  ({test_time/len(test_preds)*1000:.0f} ms/sample)")

## 11. Translation examples + error analysis

In [ ]:
def error_note(ref, pred):
    if pred.lower().strip() == ref.lower().strip():
        return "exact match"
    overlap = len(set(ref.lower().split()) & set(pred.lower().split())) / max(len(ref.split()), 1)
    if overlap < 0.2: return "low overlap — meaning drift"
    if overlap > 0.6: return "partial match — minor differences"
    return "moderate overlap — missing/added content"

examples = pd.DataFrame({
    "Source_id": dev_df[ID_COL].iloc[:10].astype(int),
    "Sanskrit": dev_df["source"].iloc[:10].values,
    "Reference_en": dev_df["target"].iloc[:10].values,
    "Prediction_en": dev_preds[:10],
    "Error_analysis": [error_note(r, p) for r, p in zip(dev_df["target"].iloc[:10], dev_preds[:10])],
})
display(examples)

## 12. Metrics table + report draft

In [ ]:
metrics = pd.DataFrame([
    {"Metric": "Pretrained model", "Value": MODEL_NAME},
    {"Metric": "LoRA rank", "Value": LORA_R},
    {"Metric": "Dev BLEU", "Value": f"{dev_bleu:.4f}"},
    {"Metric": "Dev BERTScore F1", "Value": f"{dev_bert:.4f}"},
    {"Metric": "Dev inference (s)", "Value": f"{dev_time:.1f}"},
    {"Metric": "Test inference (s)", "Value": f"{test_time:.1f}"},
    {"Metric": "Total parameters", "Value": f"{total_params:,}"},
    {"Metric": "Trainable parameters", "Value": f"{trainable_params:,}"},
    {"Metric": "Beam width", "Value": NUM_BEAMS},
])
display(metrics)
metrics.to_csv(METRICS_DIR / "final_metrics.csv", index=False)

test_bleu_str = f"{test_bleu:.4f}" if test_bleu is not None else "N/A"
test_bert_str = f"{test_bert:.4f}" if test_bert is not None else "N/A"

report = f"""# Sanskrit→English NMT — Report Draft

## Introduction
Sanskrit→English MT on ~10k pairs using **{MODEL_NAME}** fine-tuned with LoRA (rank {LORA_R}).
NLLB-200 officially supports Sanskrit as **`{SRC_LANG}`**; English target is **`{TGT_LANG}`**.

## Architecture
Encoder–decoder Transformer (NLLB/M2M100). Multi-head attention. Decoding: beam search (beams={NUM_BEAMS}).
Only LoRA adapters on `q_proj`/`v_proj` are trained ({trainable_params:,} / {total_params:,} params).

## Training
- Data: professor CSVs aligned on Source_id; whitespace normalization only
- Optimizer: AdamW, lr={LEARNING_RATE}, weight decay={WEIGHT_DECAY}
- Batch {TRAIN_BATCH_SIZE} × grad accum {GRAD_ACCUM_STEPS}; early stopping (patience={EARLY_STOPPING_PATIENCE}) on dev BLEU
- Max lengths: source={MAX_SOURCE_LENGTH}, target={MAX_TARGET_LENGTH}
- Device: {DEVICE}; LoRA full-precision (no QLoRA on macOS)

## Results
| Metric | Value |
|--------|-------|
| Dev BLEU | {dev_bleu:.4f} |
| Dev BERTScore F1 | {dev_bert:.4f} |
| Test BLEU (local) | {test_bleu_str} |
| Test BERTScore F1 (local) | {test_bert_str} |
| Dev inference | {dev_time:.1f} s |
| Test inference | {test_time:.1f} s |

## Disclosure
Pretrained **{MODEL_NAME}** (Meta NLLB-200). Fine-tuning data: provided train/dev only.

## Discussion
Sanskrit is morphologically rich and low-resource; mixed Devanagari/Latin tokens (technical terms) remain hard.
Limitations: 10k pairs, no external monolingual data, MPS without fp16.

## References
1. NLLB Team (2022). arXiv:2207.04672
2. Hu et al. (2021). LoRA. arXiv:2106.09685
3. Papineni et al. (2002). BLEU.
4. Zhang et al. (2020). BERTScore.
"""
(PROJECT_DIR / "REPORT_DRAFT.md").write_text(report, encoding="utf-8")
print("Saved REPORT_DRAFT.md")

---
**Submit:** `submission.csv` · this notebook · `REPORT_DRAFT.md` → PDF